<!-- AI-og-helse: colab-kontrakt v1 -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/uke02-klassisk-ml/01-klassisk-ml-101.ipynb)

## Colab-kjøring

- **Anbefalt runtime:** CPU
- **Forventet kjøretid:** 5-15 min
- **Datamønster:** `synthetic-or-inline`

**Krav før kjøring:**
- API: ikke nødvendig
- Data: syntetisk eller innebygd i notebooken
- GPU: ikke nødvendig

**Felles konvensjon:** Kjør setup-cellen rett under først. I Colab hentes hemmeligheter fra **Secrets** med `userdata.get(...)`; lokalt brukes miljøvariabler eller `.env`.

Kjør cellene ovenfra og ned. Setup-cellen installerer bare ekstra pakker når notebooken åpnes i Colab.


In [ ]:
# AI-og-helse: Colab bootstrap v1
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_PACKAGES = []
COLAB_DATA_MODE = 'synthetic-or-inline'


def _has_import(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is not None


def ensure_packages(packages=NOTEBOOK_PACKAGES):
    """Install only notebook-specific packages when running in Colab."""
    if not IN_COLAB:
        return

    missing = [package for package, import_name in packages if not _has_import(import_name)]
    if missing:
        print("Installerer Colab-pakker:", ", ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Alle notebook-spesifikke Colab-pakker er tilgjengelige.")


def get_secret(name: str, *aliases: str):
    """Read secrets from environment/.env locally or Colab Secrets in Colab."""
    for key in (name, *aliases):
        value = os.getenv(key)
        if value:
            os.environ[name] = value
            return value

    if IN_COLAB:
        try:
            from google.colab import userdata
        except Exception:
            userdata = None

        if userdata is not None:
            for key in (name, *aliases):
                try:
                    value = userdata.get(key)
                except Exception:
                    value = None
                if value:
                    os.environ[name] = value
                    return value

    return None


def mount_drive_if_needed():
    """Mount Google Drive explicitly in notebooks that need persistent artifacts."""
    if not IN_COLAB:
        return None
    from google.colab import drive

    drive.mount("/content/drive")
    return Path("/content/drive/MyDrive")


ensure_packages()
print("Miljø:", "Google Colab" if IN_COLAB else "lokalt")
print("Datamønster:", COLAB_DATA_MODE)


# 🎓 Klassisk Maskinlæring 101
## En enkel introduksjon for helsepersonell

`01-klassisk-ml-101.ipynb`

### 📚 Hva lærer du i denne notebooken?
1. **Hva er maskinlæring?** - Enkle forklaringer uten teknisk sjargong
2. **Supervised vs. Unsupervised** - To hovedtyper med helseeksempler
3. **Beslutningstrær** - Hvordan datamaskiner "tenker" som leger
4. **Evaluering** - Hvordan vet vi om modellen er god?
5. **Grunnleggende Python** - Bare det du trenger for å forstå

### ⏱️ Tidsbruk: 20-30 minutter



### Felles imports

Colab-oppsettet er allerede håndtert i standardcellen øverst. Denne cellen laster bare bibliotekene som brukes videre i notebooken.


In [ ]:
# Felles imports for notebooken
import json
import os
import sys
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = ["DejaVu Sans", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("Miljø:", "Google Colab" if IN_COLAB else "lokalt")
print('✅ Imports klare.')


---

## 🤔 Del 1: Hva er maskinlæring egentlig?

### Tenk deg dette scenariet:
Du er en erfaren sykepleier på en medisinsk avdeling. Gjennom årene har du sett tusenvis av pasienter og begynt å merke mønstre:

- Pasienter med høy feber + hvite blodceller > 15 + hodepine → ofte bakteriell infeksjon
- Pasienter med brystsmerter + kortpustethet + høy troponin → mulig hjerteinfarkt
- Pasienter med tretthet + høy blodsukker + tørste → sannsynligvis diabetes

**Dette ER maskinlæring!** Du lærer av erfaring og bruker mønstre til å gjøre prediksjoner.

### Forskjellen er:
- **Du** bruker hjernen din og årvis erfaring
- **Maskinen** bruker matematikk og tusenvis av eksempler

Men prinsippet er det samme: **lære av data for å gjøre forutsigelser**.

## 📊 Del 2: To hovedtyper maskinlæring

### 🎯 Supervised Learning ("Med fasit")
**Analogi:** Som å lære med en lærebok hvor svarene står bak i boka

**Eksempel fra medisin:**
- **Input:** Symptomer, labverdier, alder
- **Output:** Diagnose (som vi allerede kjenner)
- **Læring:** Maskinen lærer sammenhengen mellom symptomer og diagnoser
- **Bruk:** Predikere diagnose for nye pasienter

**Konkrete eksempler:**
- Predikere diabetes basert på blodsukker og BMI
- Gjenkjenne hudkreft på bilder
- Forutsi risiko for hjerteinfarkt

---

### 🔍 Unsupervised Learning ("Uten fasit")
**Analogi:** Som å være detektiv og lete etter skjulte mønstre

**Eksempel fra medisin:**
- **Input:** Bare symptomddata, ingen kjent diagnose
- **Læring:** Finn grupper av pasienter som ligner på hverandre
- **Bruk:** Oppdage nye sykdomstyper eller pasientgrupper

**Konkrete eksempler:**
- Finne undergrupper av diabetespasienter
- Oppdage ukjente bivirkninger av medisiner
- Identifisere nye pandemi-mønstre

## 🌳 Del 3: Beslutningstrær - Maskinens "kliniske resonnering"

### Hvordan tenker du som helsepersonell?
```
Pasient kommer med brystsmerter
↓
Er pasienten over 50 år?
├─ JA → Høy risiko, sjekk EKG og troponin
└─ NEI → Er det anstrengelsesutløst?
           ├─ JA → Moderat risiko, stress-test
           └─ NEI → Lav risiko, observasjon
```

### Slik "tenker" et beslutningstre:
```
Pasient med ukjent diabetesstatus
↓
Er HbA1c > 6.5%?
├─ JA → 90% sannsynlighet for diabetes
└─ NEI → Er BMI > 30?
           ├─ JA → Er alder > 45?
           │      ├─ JA → 60% sannsynlighet
           │      └─ NEI → 20% sannsynlighet
           └─ NEI → 5% sannsynlighet
```

### Hvorfor er dette bra?
- ✅ **Lett å forstå** - følger klinisk logikk
- ✅ **Lett å forklare** til pasienter og kolleger
- ✅ **Transparent** - du ser hvordan beslutningen tas
- ✅ **Rask** - få spørsmål gir svar

## 🧮 Del 4: Grunnleggende Python (kun det nødvendige!)

Ikke bekymre deg - du trenger ikke å kunne programmere! Her er bare det du trenger for å forstå:

In [2]:
# Dette er en kommentar - Python ignorerer alt etter #
# La oss lage enkle eksempler som du forstår:

# Variabler - som å skrive på en lapp
pasient_alder = 65
pasient_navn = "Kari"
har_diabetes = True  # True = sant, False = usant

print(f"Pasienten {pasient_navn} er {pasient_alder} år gammel")

# Enkle beregninger
bmi = 85 / (1.75 ** 2)  # vekt / høyde^2
print(f"BMI: {bmi:.1f}")

Pasienten Kari er 65 år gammel
BMI: 27.8


In [3]:
# Lister - som en pasientliste
blodsukkerverdier = [5.2, 7.8, 6.1, 12.3, 4.9]
pasientnavn = ["Anna", "Bjørn", "Cecilie", "David", "Emma"]

print("Første pasient:", pasientnavn[0])  # Python teller fra 0!
print("Første blodsukkermåling:", blodsukkerverdier[0])

# Hvor mange pasienter?
antall_pasienter = len(pasientnavn)
print(f"Vi har {antall_pasienter} pasienter")

Første pasient: Anna
Første blodsukkermåling: 5.2
Vi har 5 pasienter


In [4]:
# If-setninger - som kliniske retningslinjer
fastende_glukose = 7.5

if fastende_glukose >= 7.0:
    print("🚨 Diabetes-diagnose (≥7.0 mmol/L)")
elif fastende_glukose >= 6.1:
    print("⚠️  Nedsatt glukosetoleranse (6.1-6.9 mmol/L)")
else:
    print("✅ Normal fastende glukose (<6.1 mmol/L)")

🚨 Diabetes-diagnose (≥7.0 mmol/L)


## 📈 Del 5: Et mini-eksempel med ekte data

La oss lage et enkelt eksempel med 5 pasienter:

In [5]:
# Last inn biblioteker vi trenger (som å hente fram verktøykassa)
import pandas as pd  # For å håndtere data (som Excel)

# Lag en liten pasienttabell
pasienter = {
    'Navn': ['Anna', 'Bjørn', 'Cecilie', 'David', 'Emma'],
    'Alder': [45, 67, 34, 55, 72],
    'BMI': [22.5, 31.2, 24.1, 28.7, 26.3],
    'Fastende_glukose': [5.1, 8.2, 4.9, 7.8, 6.5],
    'Diabetes': ['Nei', 'Ja', 'Nei', 'Ja', 'Usikker']
}

# Gjør om til en tabell (DataFrame)
df = pd.DataFrame(pasienter)
print("📋 Vår pasienttabell:")
print(df)

📋 Vår pasienttabell:
      Navn  Alder   BMI  Fastende_glukose Diabetes
0     Anna     45  22.5               5.1      Nei
1    Bjørn     67  31.2               8.2       Ja
2  Cecilie     34  24.1               4.9      Nei
3    David     55  28.7               7.8       Ja
4     Emma     72  26.3               6.5  Usikker


In [6]:
# La oss se på enkle sammenhenger
print("🔍 Enkle observasjoner:")
print()

# Hvem har høy BMI?
hoy_bmi = df[df['BMI'] > 30]
print("Pasienter med BMI > 30:")
for i, row in hoy_bmi.iterrows():
    print(f"  - {row['Navn']}: BMI {row['BMI']}, Diabetes: {row['Diabetes']}")

print()

# Hvem har høy blodukker?
hoy_sukker = df[df['Fastende_glukose'] > 7.0]
print("Pasienter med fastende glukose > 7.0:")
for i, row in hoy_sukker.iterrows():
    print(f"  - {row['Navn']}: {row['Fastende_glukose']} mmol/L, Diabetes: {row['Diabetes']}")

🔍 Enkle observasjoner:

Pasienter med BMI > 30:
  - Bjørn: BMI 31.2, Diabetes: Ja

Pasienter med fastende glukose > 7.0:
  - Bjørn: 8.2 mmol/L, Diabetes: Ja
  - David: 7.8 mmol/L, Diabetes: Ja


## 🎯 Del 6: Evaluering - Hvor god er modellen?

### De fire mulige utfallene (Confusion Matrix):

```
                    VIRKELIGHET
                 Har IKKE diabetes  |  Har diabetes
MODELL  Nei    |   ✅ RIKTIG      |   ❌ FEIL
SIER    Ja     |   ❌ FEIL        |   ✅ RIKTIG
```

### Medisinsk terminologi:
- **Sant Negative (SN):** Friske som testes negative ✅
- **Sant Positive (SP):** Syke som testes positive ✅  
- **Falsk Positive (FP):** Friske som testes positive ❌ ("falsk alarm")
- **Falsk Negative (FN):** Syke som testes negative ❌ ("gått glipp av")

### Viktige metrikker:

**🎯 Sensitivitet = SP / (SP + FN)**
- "Hvor mange syke fanger vi opp?"
- Eksempel: 85% sensitivitet = vi finner 85 av 100 diabetikere

**🎯 Spesifisitet = SN / (SN + FP)**  
- "Hvor mange friske identifiserer vi korrekt?"
- Eksempel: 90% spesifisitet = vi sier korrekt at 90 av 100 friske IKKE har diabetes

In [7]:
# Et enkelt eksempel på evaluering
print("📊 Eksempel på modell-evaluering:")
print()
print("Vi testet modellen på 100 nye pasienter:")
print()

# Resultater
sant_positive = 40    # Diabetikere som ble oppdaget
falsk_negative = 10   # Diabetikere som ble savnet
sant_negative = 45    # Friske som ble identifisert korrekt
falsk_positive = 5    # Friske som fikk feil diagnose

# Beregn metrikker
sensitivitet = sant_positive / (sant_positive + falsk_negative)
spesifisitet = sant_negative / (sant_negative + falsk_positive)

print(f"📈 Sensitivitet: {sensitivitet:.1%}")
print(f"   → Vi oppdaget {sant_positive} av {sant_positive + falsk_negative} diabetikere")
print()
print(f"📈 Spesifisitet: {spesifisitet:.1%}")
print(f"   → Vi identifiserte korrekt {sant_negative} av {sant_negative + falsk_positive} friske")
print()
print("🤔 Hva betyr dette i praksis?")
print(f"   - {falsk_negative} diabetikere får ikke behandling (alvorlig!)")
print(f"   - {falsk_positive} friske får unødvendig bekymring og testing")

📊 Eksempel på modell-evaluering:

Vi testet modellen på 100 nye pasienter:

📈 Sensitivitet: 80.0%
   → Vi oppdaget 40 av 50 diabetikere

📈 Spesifisitet: 90.0%
   → Vi identifiserte korrekt 45 av 50 friske

🤔 Hva betyr dette i praksis?
   - 10 diabetikere får ikke behandling (alvorlig!)
   - 5 friske får unødvendig bekymring og testing


## ⚖️ Del 7: Avveininger i medisin

### Hva er verst - false positive eller false negative?

**Det kommer an på situasjonen!**

#### False Negative er verst ved:
- **Cancer-screening** 🎗️ (går glipp av kreft)
- **Hjerteinfarkt** 💔 (sender hjem pasient som trenger akutt behandling)
- **Sepsis** 🦠 (livstruende infeksjon)

→ **Prioriter høy sensitivitet**

#### False Positive er verst ved:
- **Sjeldne sykdommer** 🔬 (mange får unødvendig angst)
- **Dyre oppfølginger** 💰 (sløser ressurser)
- **Invasive prosedyrer** ⚕️ (unødvendig risiko)

→ **Prioriter høy spesifisitet**

### For diabetes-screening:
- Moderat alvorlighet
- God behandling tilgjengelig  
- **Balanse:** Vi ønsker både høy sensitivitet og spesifisitet

## 🎓 Del 8: Sammendrag - Hva har du lært?

### 🧠 Maskinlæring:
- Er som klinisk erfaring, bare med matematikk istedenfor intuisjon
- Lærer mønstre fra store datamengder
- Kan støtte (ikke erstatte) klinisk dømmekraft

### 📊 To hovedtyper:
- **Supervised:** Lærer med "fasit" (predikere diabetes fra symptomer)
- **Unsupervised:** Finner skjulte mønstre (oppdage pasientgrupper)

### 🌳 Beslutningstrær:
- Følger klinisk logikk (hvis-så-resonnering)
- Lett å forstå og forklare
- Transparent beslutningsprosess

### 🎯 Evaluering:
- **Sensitivitet:** Hvor mange syke fanger vi?
- **Spesifisitet:** Hvor mange friske identifiserer vi korrekt?
- **Avveining:** Hva er verst - false positive eller false negative?

### 🔮 Du er nå klar for:
- Den mer avanserte notebooken `02-fra-symptom-til-diagnose.ipynb`
- Å forstå hvordan AI kan brukes i din kliniske praksis
- Å delta i diskusjoner om AI i helsevesenet

---

## 🚀 Neste steg:
Gå nå til `02-fra-symptom-til-diagnose.ipynb` hvor vi bygger en ekte diabetes-prediksjonsmodell med 1000 pasienter!

**Husk:** Du trenger ikke å forstå all koden - fokuser på konseptene og hvordan de kan brukes i praksis. 🏥